In [1]:
print("Let's Start")


Let's Start


In [2]:
import pandas as td
from sklearn.model_selection import train_test_split
df = td.read_csv('/mnt/WindowsD/Projects/commentlens/data/interim/comments.csv' , index_col=0)

df = df.iloc[: , :2]

x = df["comments"].astype(str)
labels = df["category"].astype(str).str.strip().str.lower()
label_mapping = {"positive": 1, "negative": 2, "neutral": 0}
y = labels.map(label_mapping)
if y.isna().any():
    invalid = sorted(labels[y.isna()].unique())
    raise ValueError(f"Unsupported categories: {invalid}")

x_train_, x_test_, y_train, y_test = train_test_split(
        x,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )


In [3]:
x.shape

(36662,)

In [3]:
y.isna().sum()

np.int64(0)

In [3]:
from lightgbm import LGBMClassifier
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
vectorizer = CountVectorizer(
    max_features=6000,
    ngram_range=(1, 2),
    dtype=np.float32,
)

x_train_tf = vectorizer.fit_transform(x_train_)
x_test_tf = vectorizer.transform(x_test_)

In [4]:
x_tran_df = td.DataFrame(x_train_tf.toarray())
x_test_df = td.DataFrame(x_test_tf.toarray())


In [5]:
x_tran_df.shape , x_test_df.shape

((29329, 6000), (7333, 6000))

In [6]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
x_train, y_train = smote.fit_resample(
    x_tran_df,
    y_train,
)
x_test , y_test = smote.fit_resample(x_test_df , y_test)



In [10]:
y_test.isna().sum()

np.int64(0)

In [7]:
merged_df = td.concat([x_train , x_test] , axis=0)
merged_df.shape

(47310, 6000)

In [8]:
y = td.concat([y_train , y_test])

In [ ]:
merged_df_with_y = td.concat([merged_df , y] , axis=0)

In [19]:
merged_df_with_y.shape

(94620, 6001)

In [15]:
import pickle

pickle.dump(merged_df_with_y ,open('/mnt/WindowsD/Projects/commentlens/data/interim/vectorized_df.pkl' , 'wb'))

In [9]:
import pickle

pickle.dump(vectorizer ,open('/mnt/WindowsD/Projects/commentlens/model/vectorizer.pkl' , 'wb'))

In [ ]:
merged_df_with_y.to_csv('/mnt/WindowsD/Projects/commentlens/data/interim/vectorized_df.pkl')

In [18]:
y.shape

(47310,)

In [4]:
import pickle

df = pickle.load(open('/mnt/WindowsD/Projects/commentlens/data/interim/vectorized_df.pkl' , 'rb'))

In [ ]:
x =df.iloc[:,:-1]
y = df.iloc[:,-1]
x.head() , y.head()

(     0    1    2    3    4    5    6    7    8    9  ...  5990  5991  5992  \
 0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   0.0   
 1  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   0.0   
 2  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   0.0   
 3  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   1.0   0.0   0.0   
 4  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   0.0   
 
    5993  5994  5995  5996  5997  5998  5999  
 0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
 1   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
 2   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
 3   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
 4   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
 
 [5 rows x 6000 columns],
 0    2.0
 1    0.0
 2    0.0
 3    1.0
 4    0.0
 Name: category, dtype: float64)

In [2]:
from automac.autotune import ClassificationTuner

ct = ClassificationTuner(x ,y)
score = ct.tune(model_keys=['lgb'])
score

/mnt/WindowsD/Projects/commentlens/.commentlens/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Error loading stopwords: The read operation timed out


NameError: name 'x' is not defined

In [5]:
import os
import zipfile
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ==========================================
# # 1. Unzip & Load Saved Model + Tokeniser
# # ==========================================
# MODEL_DIR = "./commentlens_muril_final"
# ZIP_PATH = "commentlens_muril_final.zip"

# model_path = os.path.abspath(MODEL_DIR)
# zip_path = os.path.abspath(ZIP_PATH)

# if not os.path.isfile(os.path.join(model_path, "config.json")) and os.path.isfile(zip_path):
#     os.makedirs(model_path, exist_ok=True)
#     with zipfile.ZipFile(zip_path, "r") as zf:
#         zf.extractall(model_path)

# if os.path.isfile(os.path.join(model_path, "config.json")):
#     model_config_path = os.path.join(model_path, "config.json")
# else:
#     model_config_path = None
#     for root, _, files in os.walk(model_path):
#         if "config.json" in files:
#             model_config_path = os.path.join(root, "config.json")
#             break

# if model_config_path is None:
#     raise FileNotFoundError(
#         f"Could not find config.json in {model_path} or its ZIP archive. "
#         f"Check that the model files exist and ZIP_PATH points to the correct archive."
#     )

model_path = '/mnt/WindowsD/Projects/commentlens/model' #os.path.dirname(model_config_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
model.eval()

ID_TO_LABEL = {0: "Neutral", 1: "Positive", 2: "Negative"}


# ==========================================
# 2. Batched Comment Prediction Helper
# ==========================================
@torch.inference_mode()
def predict_comments(comments, batch_size=64, max_len=128):
    if not comments:
        return []

    cleaned = [str(c).strip() for c in comments]
    results = []

    for start in range(0, len(cleaned), batch_size):
        batch_texts = cleaned[start : start + batch_size]
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt",
        ).to(device)

        if device.type == "cuda":
            with torch.amp.autocast("cuda", dtype=torch.float16):
                logits = model(**inputs).logits
        else:
            logits = model(**inputs).logits

        probs = torch.softmax(logits.float(), dim=-1).cpu().numpy()

        for text, prob_row in zip(batch_texts, probs):
            pred_id = int(np.argmax(prob_row))
            results.append(
                {
                    "comment": text,
                    "sentiment": ID_TO_LABEL[pred_id],
                    "confidence": round(float(prob_row[pred_id]), 4),
                    "neutral_prob": round(float(prob_row[0]), 4),
                    "positive_prob": round(float(prob_row[1]), 4),
                    "negative_prob": round(float(prob_row[2]), 4),
                }
            )

    return results


# ==========================================
# 3. Video-Level Summary (For CommentLens Extension)
# ==========================================
def analyze_video_comments(comments, batch_size=64):
    predictions = predict_comments(comments, batch_size=batch_size)
    total = len(predictions)
    if total == 0:
        return {"total_comments": 0}

    counts = {"Positive": 0, "Negative": 0, "Neutral": 0}
    for item in predictions:
        counts[item["sentiment"]] += 1

    percentages = {
        label: round((count / total) * 100, 2) for label, count in counts.items()
    }

    return {
        "total_comments": total,
        "counts": counts,
        "percentages": percentages,
        "predictions": predictions,
    }


# ==========================================
# 4. Test on Sample English + Hinglish YouTube Comments
# ==========================================
sample_comments = [
    "Bhai explanation ekdum clear thi, poora concept samajh aa gaya!",
    "Complete waste of time, clickbait thumbnail and zero useful info.",
    "Part 2 kab tak upload hoga?",
    "Editing is top notch but background music is way too loud.",
    "Ye trick sach mein kaam karti hai, thanks a lot bro!",
    "Kitna bakwas audio hai, kuch sunai hi nahi de raha.",
    "Timestamp 04:20 par jo formula bataya wo exam mein aaya tha.",
    "Not bad, decent overview for beginners.",
]

report = analyze_video_comments(sample_comments)

print("=== Video Sentiment Summary ===")
print(f"Total Comments: {report['total_comments']}")
print(f"Counts        : {report['counts']}")
print(f"Percentages   : {report['percentages']}\n")

df_results = pd.DataFrame(report["predictions"])
print(df_results[["comment", "sentiment", "confidence", "positive_prob", "negative_prob", "neutral_prob"]].to_string(index=False))

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1267.17it/s]


=== Video Sentiment Summary ===
Total Comments: 8
Counts        : {'Positive': 4, 'Negative': 1, 'Neutral': 3}
Percentages   : {'Positive': 50.0, 'Negative': 12.5, 'Neutral': 37.5}

                                                          comment sentiment  confidence  positive_prob  negative_prob  neutral_prob
  Bhai explanation ekdum clear thi, poora concept samajh aa gaya!  Positive      0.7844         0.7844         0.1277        0.0879
Complete waste of time, clickbait thumbnail and zero useful info.  Positive      0.8045         0.8045         0.1621        0.0334
                                      Part 2 kab tak upload hoga?   Neutral      0.9058         0.0385         0.0557        0.9058
       Editing is top notch but background music is way too loud.  Positive      0.7591         0.7591         0.2158        0.0251
             Ye trick sach mein kaam karti hai, thanks a lot bro!  Positive      0.8127         0.8127         0.1365        0.0508
              Kitna bakwas

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

df = pd.read_csv("/mnt/WindowsD/Projects/commentlens/data/interim/comments.csv")[["comments", "category"]].dropna()
x = df["comments"].astype(str)
y = df["category"].astype(str).str.strip().str.lower().map({"neutral": 0, "positive": 1, "negative": 2})

_, x_test, _, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

preds = predict_comments(x_test.tolist(), batch_size=128)
label_to_id = {"Neutral": 0, "Positive": 1, "Negative": 2}
y_pred = [label_to_id[p["sentiment"]] for p in preds]

print(classification_report(y_test, y_pred, target_names=["Neutral (0)", "Positive (1)", "Negative (2)"]))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

KeyboardInterrupt: 